# Test statistique sur les données tomate (round 0, jour -2, -1)

In [1]:

# ── Core ──────────────────────────────────────────────────────────────────────
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Statistical tests ─────────────────────────────────────────────────────────
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf, q_stat
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from scipy import stats
from scipy.signal import periodogram



## 1. Load data & visualisation

We plot the data to get a feel for the data range, scale, visible trends, volatilty.
We also calculate the log returns.

In [5]:
df1 = pd.read_csv(r'C:\Users\fogat\Desktop\imc_trading\imc_trading\TUTORIAL_ROUND_1\données\prices_round_0_day_-1.csv', sep=';')
df2 = pd.read_csv(r'C:\Users\fogat\Desktop\imc_trading\imc_trading\TUTORIAL_ROUND_1\données\prices_round_0_day_-2.csv', sep=';')

dftom = pd.concat([df2[df2['product'] == 'TOMATOES'], df1[df1['product'] == 'TOMATOES']], ignore_index=True)


df1 = df1[df1['product'] == 'TOMATOES']
df2 = df2[df2['product'] == 'TOMATOES']

log_returns = np.log(dftom['mid_price']).diff().dropna()

colnames = dftom.columns.tolist()

for col in colnames:
    if 'bid' in col:
        print(f'{col} range: [{dftom[col].min()}, {dftom[col].max()}]')
        print(f'{col} std dev: {dftom[col].std()}')
    elif 'ask' in col:
        print(f'{col} range: [{dftom[col].min()}, {dftom[col].max()}]')
        print(f'{col} std dev: {dftom[col].std()}')
    elif 'mid_price' in col:
        print(f'{col} range: [{dftom[col].min()}, {dftom[col].max()}]')
        print(f'{col} std dev: {dftom[col].std()}')



bid_price_1 range: [4941, 5032]
bid_price_1 std dev: 19.733855863803267
bid_volume_1 range: [2, 12]
bid_volume_1 std dev: 1.7922833520346886
bid_price_2 range: [4940, 5027]
bid_price_2 std dev: 19.70069130612926
bid_volume_2 range: [3, 25]
bid_volume_2 std dev: 3.892412069795033
bid_price_3 range: [4942.0, 5025.0]
bid_price_3 std dev: 19.345590335229684
bid_volume_3 range: [5.0, 25.0]
bid_volume_3 std dev: 3.2855601823087004
ask_price_1 range: [4950, 5042]
ask_price_1 std dev: 19.799156352408385
ask_volume_1 range: [1, 12]
ask_volume_1 std dev: 1.7931093830447093
ask_price_2 range: [4956, 5043]
ask_price_2 std dev: 19.756535588822846
ask_volume_2 range: [5, 25]
ask_volume_2 std dev: 3.8835866505414316
ask_price_3 range: [4957.0, 5041.0]
ask_price_3 std dev: 20.012692208273208
ask_volume_3 range: [15.0, 25.0]
ask_volume_3 std dev: 3.1945177113201626
mid_price range: [4946.5, 5036.0]
mid_price std dev: 19.747055003048928


In [6]:
#Raw price plotly plot (bid ask, mid price)
#day -1 and day -2 separately
fig = px.line(df2, x='timestamp', y=['bid_price_1', 'ask_price_1', 'mid_price'], title='Bid, Ask, Mid Prices of TOMATOES Over Time (day -2)')
fig.update_layout(xaxis_title='Timestamp', yaxis_title='Price')
fig.show()

fig = px.line(df1, x='timestamp', y=['bid_price_1', 'ask_price_1', 'mid_price'], title='Bid, Ask, Mid Prices of TOMATOES Over Time (day -1)')
fig.update_layout(xaxis_title='Timestamp', yaxis_title='Price')
fig.show()

#Log returns plotly plot

fig = px.line(log_returns, x=log_returns.index, y=log_returns.values, title='Log Returns of TOMATOES Mid Price Over Time')
fig.update_layout(xaxis_title='Timestamp', yaxis_title='Log Return')
fig.show()

#rolling mean and std dev plotly plot

rolling_window = 50
rolling_mean = log_returns.rolling(window=rolling_window).mean()  
rolling_std = log_returns.rolling(window=rolling_window).std()
fig = go.Figure()
fig.add_trace(go.Scatter(x=log_returns.index, y=log_returns.values, mode='lines', name='Log Returns'))
fig.add_trace(go.Scatter(x=rolling_mean.index, y=rolling_mean.values, mode='lines', name=f'{rolling_window}-Period Rolling Mean'))
fig.add_trace(go.Scatter(x=rolling_std.index, y=rolling_std.values, mode='lines', name=f'{rolling_window}-Period Rolling Std Dev'))
fig.update_layout(title='Log Returns with Rolling Mean and Std Dev', xaxis_title='Timestamp', yaxis_title='Value')
fig.show()

### 🔍 Interpretation guide

| What you see | What it means |
|---|---|
| Rolling mean drifts up/down | Possible non-stationarity or drift |
| Rolling mean stays roughly flat | Mean-stationary |
| Returns cluster into volatile/calm periods | Volatility clustering → ARCH effects |
| Returns look uniformly noisy | Homoskedastic — simpler models may suffice |
| Price bounces around a level | Mean-reversion candidate |

> **Note:** Visual inspection is never conclusive — it primes your expectations before the formal tests below.

### Stat tests

In [7]:
def run_stationarity_tests(series, series_name='Series', verbose=True):
    """
    Runs ADF and KPSS tests and prints a formatted report.
    Returns a dict of results.
    """
    results = {}

    # ── ADF ───────────────────────────────────────────────────────────────────
    # autolag='AIC' automatically picks the number of lagged differences
    # to include so residuals are white noise (avoids over/under-differencing)
    adf_stat, adf_p, adf_lags, adf_nobs, adf_cv, _ = adfuller(series.dropna(), autolag='AIC')
    results['ADF stat']   = adf_stat
    results['ADF p-val']  = adf_p
    results['ADF lags']   = adf_lags

    # ── KPSS ──────────────────────────────────────────────────────────────────
    # regression='c' tests level stationarity
    # regression='ct' tests trend stationarity
    kpss_stat, kpss_p, kpss_lags, kpss_cv = kpss(series.dropna(), regression='c', nlags='auto')
    results['KPSS stat']  = kpss_stat
    results['KPSS p-val'] = kpss_p

    if verbose:
        print(f"{'='*55}")
        print(f"  Stationarity Tests: {series_name}")
        print(f"{'='*55}")
        print(f"\n{'─'*30} ADF Test {'─'*15}")
        print(f"  H₀: Unit root exists (non-stationary)")
        print(f"  Test statistic : {adf_stat:.4f}")
        print(f"  p-value        : {adf_p:.4f}")
        print(f"  Lags used      : {adf_lags}")
        for level, cv in adf_cv.items():
            sig = '← REJECT H₀' if adf_stat < cv else ''
            print(f"  Critical value {level}: {cv:.4f}  {sig}")

        print(f"\n{'─'*30} KPSS Test {'─'*14}")
        print(f"  H₀: Series is stationary")
        print(f"  Test statistic : {kpss_stat:.4f}")
        print(f"  p-value        : {kpss_p:.4f}")
        for level, cv in kpss_cv.items():
            sig = '← REJECT H₀' if kpss_stat > cv else ''
            print(f"  Critical value {level}: {cv:.4f}  {sig}")

        print(f"\n{'─'*55}")
        adf_stationary  = adf_p  < 0.05
        kpss_stationary = kpss_p > 0.05
        if adf_stationary and kpss_stationary:
            verdict = '✅ Both tests agree: STATIONARY'
        elif not adf_stationary and not kpss_stationary:
            verdict = '❌ Both tests agree: NON-STATIONARY'
        elif adf_stationary and not kpss_stationary:
            verdict = '⚠️  Conflict — possible TREND-STATIONARY'
        else:
            verdict = '⚠️  Conflict — insufficient evidence'
        print(f"  Verdict: {verdict}")
        print(f"{'='*55}\n")

    return results


# Run on mid price levels and on returns (for day -1 and day -2 separately)
log_returns1 = np.log(df1['mid_price']).diff().dropna()
log_returns2 = np.log(df2['mid_price']).diff().dropna()
mid_price_levels1 = df1['mid_price']
mid_price_levels2 = df2['mid_price']

#day -1
print("\n" + "#"*60)
print("  PRICE LEVELS day -1")
print("#"*60)
res_price = run_stationarity_tests(mid_price_levels1, 'Price levels')

print("\n" + "#"*60)
print("  LOG RETURNS day -1")
print("#"*60)
res_returns = run_stationarity_tests(log_returns1, 'Log returns')

#day -2
print("\n" + "#"*60)
print("  PRICE LEVELS day -2")
print("#"*60)
res_price = run_stationarity_tests(mid_price_levels2, 'Price levels')

print("\n" + "#"*60)
print("  LOG RETURNS day -2")
print("#"*60)
res_returns = run_stationarity_tests(log_returns2, 'Log returns')


############################################################
  PRICE LEVELS day -1
############################################################


C:\Users\fogat\AppData\Local\Temp\ipykernel_22560\3766846662.py:19: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_stat, kpss_p, kpss_lags, kpss_cv = kpss(series.dropna(), regression='c', nlags='auto')


  Stationarity Tests: Price levels

────────────────────────────── ADF Test ───────────────
  H₀: Unit root exists (non-stationary)
  Test statistic : -2.0644
  p-value        : 0.2591
  Lags used      : 8
  Critical value 1%: -3.4310  
  Critical value 5%: -2.8618  
  Critical value 10%: -2.5669  

────────────────────────────── KPSS Test ──────────────
  H₀: Series is stationary
  Test statistic : 13.4211
  p-value        : 0.0100
  Critical value 10%: 0.3470  ← REJECT H₀
  Critical value 5%: 0.4630  ← REJECT H₀
  Critical value 2.5%: 0.5740  ← REJECT H₀
  Critical value 1%: 0.7390  ← REJECT H₀

───────────────────────────────────────────────────────
  Verdict: ❌ Both tests agree: NON-STATIONARY


############################################################
  LOG RETURNS day -1
############################################################


C:\Users\fogat\AppData\Local\Temp\ipykernel_22560\3766846662.py:19: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_stat, kpss_p, kpss_lags, kpss_cv = kpss(series.dropna(), regression='c', nlags='auto')


  Stationarity Tests: Log returns

────────────────────────────── ADF Test ───────────────
  H₀: Unit root exists (non-stationary)
  Test statistic : -43.3414
  p-value        : 0.0000
  Lags used      : 7
  Critical value 1%: -3.4310  ← REJECT H₀
  Critical value 5%: -2.8618  ← REJECT H₀
  Critical value 10%: -2.5669  ← REJECT H₀

────────────────────────────── KPSS Test ──────────────
  H₀: Series is stationary
  Test statistic : 0.0287
  p-value        : 0.1000
  Critical value 10%: 0.3470  
  Critical value 5%: 0.4630  
  Critical value 2.5%: 0.5740  
  Critical value 1%: 0.7390  

───────────────────────────────────────────────────────
  Verdict: ✅ Both tests agree: STATIONARY


############################################################
  PRICE LEVELS day -2
############################################################


C:\Users\fogat\AppData\Local\Temp\ipykernel_22560\3766846662.py:19: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_stat, kpss_p, kpss_lags, kpss_cv = kpss(series.dropna(), regression='c', nlags='auto')


  Stationarity Tests: Price levels

────────────────────────────── ADF Test ───────────────
  H₀: Unit root exists (non-stationary)
  Test statistic : -2.3969
  p-value        : 0.1426
  Lags used      : 9
  Critical value 1%: -3.4310  
  Critical value 5%: -2.8618  
  Critical value 10%: -2.5669  

────────────────────────────── KPSS Test ──────────────
  H₀: Series is stationary
  Test statistic : 4.4503
  p-value        : 0.0100
  Critical value 10%: 0.3470  ← REJECT H₀
  Critical value 5%: 0.4630  ← REJECT H₀
  Critical value 2.5%: 0.5740  ← REJECT H₀
  Critical value 1%: 0.7390  ← REJECT H₀

───────────────────────────────────────────────────────
  Verdict: ❌ Both tests agree: NON-STATIONARY


############################################################
  LOG RETURNS day -2
############################################################
  Stationarity Tests: Log returns

────────────────────────────── ADF Test ───────────────
  H₀: Unit root exists (non-stationary)
  Test statistic :

C:\Users\fogat\AppData\Local\Temp\ipykernel_22560\3766846662.py:19: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_stat, kpss_p, kpss_lags, kpss_cv = kpss(series.dropna(), regression='c', nlags='auto')


### Interpretation guide

**If price levels are non-stationary but returns are stationary** → the price series is **I(1)** (integrated of order 1), which is the standard result for financial prices. This means:
- You should model and analyse **returns**, not raw prices
- Mean-reversion strategies should work on *deviations from a spread or moving average*, not on the raw price

**If price levels are stationary** → prices gravitate to a long-run mean. Mean-reversion strategies on the raw price may be viable.

**Trend-stationary** → subtract a fitted linear trend from the price, then re-test. The detrended series may be stationary.

---
## Drift: OLS Trend Regression

### Theory

**Drift** is a systematic directional bias — the series tends to move up (or down) on average each step. It is distinct from a random walk, where direction at each step is unpredictable.

We test drift by fitting:  
$$p_t = \alpha + \beta \cdot t + \varepsilon_t$$

- **β** is the drift coefficient: the expected change in price per time step
- **t-statistic on β**: if |t| > 2 and p < 0.05 → drift is statistically significant
- **R²**: how much price variance is explained by the linear trend alone

We also inspect **rolling mean behaviour** — a drifting series will have a rolling mean that systematically moves in one direction.

In [8]:
# OLS regression: price ~ constant + time 

# function to run OLS regression and print results
def run_ols_drift_test(series, series_name='Series'):
    t = np.arange(len(series))
    X = add_constant(t)         # adds intercept column
    y = series.values

    ols_model  = OLS(y, X).fit()
    alpha_hat  = ols_model.params[0]
    beta_hat   = ols_model.params[1]
    beta_tstat = ols_model.tvalues[1]
    beta_pval  = ols_model.pvalues[1]
    r_squared  = ols_model.rsquared

    print("=" * 50)
    print(f"  Drift Test: OLS  {series_name} ~ α + β·t")
    print("=" * 50)
    print(f"  Intercept α      : {alpha_hat:.4f}")
    print(f"  Slope β (drift)  : {beta_hat:.6f} per step")
    print(f"  t-statistic (β)  : {beta_tstat:.4f}")
    print(f"  p-value (β)      : {beta_pval:.4f}")
    print(f"  R²               : {r_squared:.4f}")
    print()
    if abs(beta_tstat) > 2 and beta_pval < 0.05:
        direction = 'upward' if beta_hat > 0 else 'downward'
        print(f"  → Statistically significant {direction} drift detected.")
        print(f"    Expected {direction} move of {abs(beta_hat):.6f} per step")
    else:
        print("  → No statistically significant drift detected.")
    print("=" * 50)

# day -1
run_ols_drift_test(mid_price_levels1, 'Price levels day -1')
#day -2
run_ols_drift_test(mid_price_levels2, 'Price levels day -2')


  Drift Test: OLS  Price levels day -1 ~ α + β·t
  Intercept α      : 4999.2239
  Slope β (drift)  : -0.004332 per step
  t-statistic (β)  : -166.8554
  p-value (β)      : 0.0000
  R²               : 0.7358

  → Statistically significant downward drift detected.
    Expected downward move of 0.004332 per step
  Drift Test: OLS  Price levels day -2 ~ α + β·t
  Intercept α      : 4999.6306
  Slope β (drift)  : 0.001664 per step
  t-statistic (β)  : 52.7827
  p-value (β)      : 0.0000
  R²               : 0.2179

  → Statistically significant upward drift detected.
    Expected upward move of 0.001664 per step


In [10]:
# Visualize the fitted OLS line on the price levels
def plot_ols_fit(series, series_name='Series'):
    t = np.arange(len(series))
    X = add_constant(t)
    y = series.values
    ols_model  = OLS(y, X).fit()
    fitted_values = ols_model.fittedvalues

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=t, y=y, mode='lines', name='Actual'))
    fig.add_trace(go.Scatter(x=t, y=fitted_values, mode='lines', name='OLS Fit'))
    fig.update_layout(title=f'OLS Fit for {series_name}', xaxis_title='Time Step', yaxis_title='Price')
    fig.show()

plot_ols_fit(mid_price_levels1, 'Price levels day -1')
plot_ols_fit(mid_price_levels2, 'Price levels day -2')

### Interpretation guide

| Result | Meaning |
|---|---|
| Significant β, high R² | Strong deterministic trend — prices follow a predictable slope |
| Significant β, low R² | Weak drift overwhelmed by noise |
| Non-significant β | No evidence of drift — price has no long-run directional bias |
| Residuals look stationary | After removing the trend, the series may be stationary (trend-stationary process) |
| Residuals still wander | Even after detrending, stochastic non-stationarity remains |

> **Trading implication:** Significant drift in the *competition window* may be exploitable by a directional bias, but drift can be short-lived. Combine with stationarity tests.

---
## Autocorrelation: ACF, PACF & Ljung-Box Test

### Theory

**Autocorrelation at lag k** measures the linear correlation between `y_t` and `y_{t-k}`.
$$\rho_k = \frac{\text{Cov}(y_t, y_{t-k})}{\text{Var}(y_t)}$$

In an **efficient market**, returns should have zero autocorrelation — past prices contain no information about future prices.

#### ACF (Autocorrelation Function)
Plots ρ_k for all lags k. Includes the *indirect* effect of intermediate lags.

#### PACF (Partial ACF)
Plots the correlation at lag k *after removing* the effect of lags 1 through k-1.  
Useful for identifying the order of an AR process:
- If PACF cuts off at lag p → AR(p) process

#### Ljung-Box Test
Formal joint test: are *all* autocorrelations up to lag k simultaneously zero?
- **H₀:** No autocorrelation up to lag k
- Low p-value → significant autocorrelation present

**Confidence bands** in ACF/PACF plots are ±1.96/√N — spikes outside these are significant at the 5% level.

In [12]:
from plotly.subplots import make_subplots

# plotting ACF and PACF for log returns of day -1
def plot_acf_pacf(series, series_name='Series', lags=50):
    acf_vals = acf(series.dropna(), nlags=lags)
    pacf_vals = pacf(series.dropna(), nlags=lags)

    fig = make_subplots(rows=1, cols=2, subplot_titles=(f'ACF of {series_name}', f'PACF of {series_name}'))
    fig.add_trace(go.Bar(x=np.arange(len(acf_vals)), y=acf_vals, name='ACF'), row=1, col=1)
    fig.add_trace(go.Bar(x=np.arange(len(pacf_vals)), y=pacf_vals, name='PACF'), row=1, col=2)
    fig.update_layout(title=f'ACF and PACF for {series_name}', showlegend=False)
    fig.show()

plot_acf_pacf(log_returns1, 'Log returns day -1')
plot_acf_pacf(log_returns2, 'Log returns day -2')

### Ljung-Box formal test on prices and returns (for day -1 and day -2 separately) 

We test at several lag horizons to see how autocorrelation accumulates

In [15]:
lags_to_test = [1, 2, 3, 5, 10]


#day -1

lb_prices_1  = acorr_ljungbox(mid_price_levels1.dropna(), lags=lags_to_test, return_df=True)


print("=" * 55)
print("  Ljung-Box Test on Log Returns (day -1)")
print("  H₀: No autocorrelation up to lag k")
print("=" * 55)
print(f"  {'Lag':>5}  {'LB Statistic':>14}  {'p-value':>10}  {'Significant?':>14}")
print("  " + "─"*50)
for lag, row in lb_prices_1.iterrows():
    sig = '✅ YES' if row['lb_pvalue'] < 0.05 else '  no'
    print(f"  {lag:>5}  {row['lb_stat']:>14.4f}  {row['lb_pvalue']:>10.4f}  {sig:>14}")
print("=" * 55)

lb_results_1  = acorr_ljungbox(log_returns1.dropna(), lags=lags_to_test, return_df=True)


print("=" * 55)
print("  Ljung-Box Test on Log Returns (day -1)")
print("  H₀: No autocorrelation up to lag k")
print("=" * 55)
print(f"  {'Lag':>5}  {'LB Statistic':>14}  {'p-value':>10}  {'Significant?':>14}")
print("  " + "─"*50)
for lag, row in lb_results_1.iterrows():
    sig = '✅ YES' if row['lb_pvalue'] < 0.05 else '  no'
    print(f"  {lag:>5}  {row['lb_stat']:>14.4f}  {row['lb_pvalue']:>10.4f}  {sig:>14}")
print("=" * 55)

  Ljung-Box Test on Log Returns (day -1)
  H₀: No autocorrelation up to lag k
    Lag    LB Statistic     p-value    Significant?
  ──────────────────────────────────────────────────
      1       9913.2606      0.0000           ✅ YES
      2      19807.0474      0.0000           ✅ YES
      3      29683.7482      0.0000           ✅ YES
      5      49385.0485      0.0000           ✅ YES
     10      98337.8596      0.0000           ✅ YES
  Ljung-Box Test on Log Returns (day -1)
  H₀: No autocorrelation up to lag k
    Lag    LB Statistic     p-value    Significant?
  ──────────────────────────────────────────────────
      1       1701.7292      0.0000           ✅ YES
      2       1704.0745      0.0000           ✅ YES
      3       1704.6117      0.0000           ✅ YES
      5       1707.0009      0.0000           ✅ YES
     10       1710.3528      0.0000           ✅ YES


In [16]:
#day -2

lb_prices_2  = acorr_ljungbox(mid_price_levels2.dropna(), lags=lags_to_test, return_df=True)


print("=" * 55)
print("  Ljung-Box Test on Log Returns (day -2)")
print("  H₀: No autocorrelation up to lag k")
print("=" * 55)
print(f"  {'Lag':>5}  {'LB Statistic':>14}  {'p-value':>10}  {'Significant?':>14}")
print("  " + "─"*50)
for lag, row in lb_prices_2.iterrows():
    sig = '✅ YES' if row['lb_pvalue'] < 0.05 else '  no'
    print(f"  {lag:>5}  {row['lb_stat']:>14.4f}  {row['lb_pvalue']:>10.4f}  {sig:>14}")
print("=" * 55)

lb_results_2  = acorr_ljungbox(log_returns2.dropna(), lags=lags_to_test, return_df=True)


print("=" * 55)
print("  Ljung-Box Test on Log Returns (day -2)")
print("  H₀: No autocorrelation up to lag k")
print("=" * 55)
print(f"  {'Lag':>5}  {'LB Statistic':>14}  {'p-value':>10}  {'Significant?':>14}")
print("  " + "─"*50)
for lag, row in lb_results_1.iterrows():
    sig = '✅ YES' if row['lb_pvalue'] < 0.05 else '  no'
    print(f"  {lag:>5}  {row['lb_stat']:>14.4f}  {row['lb_pvalue']:>10.4f}  {sig:>14}")
print("=" * 55)

  Ljung-Box Test on Log Returns (day -2)
  H₀: No autocorrelation up to lag k
    Lag    LB Statistic     p-value    Significant?
  ──────────────────────────────────────────────────
      1       9832.1866      0.0000           ✅ YES
      2      19640.7157      0.0000           ✅ YES
      3      29426.5738      0.0000           ✅ YES
      5      48927.7908      0.0000           ✅ YES
     10      97314.5854      0.0000           ✅ YES
  Ljung-Box Test on Log Returns (day -2)
  H₀: No autocorrelation up to lag k
    Lag    LB Statistic     p-value    Significant?
  ──────────────────────────────────────────────────
      1       1701.7292      0.0000           ✅ YES
      2       1704.0745      0.0000           ✅ YES
      3       1704.6117      0.0000           ✅ YES
      5       1707.0009      0.0000           ✅ YES
     10       1710.3528      0.0000           ✅ YES


### Interpretation guide

**ACF pattern → Process type:**

| ACF shape | PACF shape | Likely model |
|---|---|---|
| Decays slowly (geometrically) | Sharp cutoff at lag p | AR(p) |
| Sharp cutoff at lag q | Decays slowly | MA(q) |
| Both decay slowly | Both decay slowly | ARMA(p,q) |
| All near zero | All near zero | White noise (efficient) |
| Decays very slowly (hyperbolic) | Same | Long memory process |
| Oscillates with period k | Peaks at multiples of k | Cyclical / seasonal |

**Ljung-Box:** If significant at short lags (5–10), the autocorrelation is economically relevant. Significant only at long lags might be noise or model mis-specification.

---
## Cycles: FFT Periodogram

### Theory

**Fourier analysis** decomposes a time series into a sum of sine and cosine waves at different frequencies. If a particular frequency dominates, the series has a **periodic cycle** at that wavelength.

The **periodogram** plots power (squared amplitude) against frequency:
- A peak at frequency f → a cycle with period T = N/f (or 1/f if frequency is in cycles-per-step)
- Flat periodogram → no dominant cycle (consistent with white noise)

We apply the FFT to:
1. **Detrended prices** — to find structural price cycles
2. **Returns** — to find return predictability cycles

> **Important:** We detrend before FFT. A strong trend injects power at all low frequencies and masks real cycles.

In [18]:
def plot_periodogram(series, series_name='Series', top_n=5):
    """
    Computes and plots the FFT periodogram, labels the top-n dominant periods.
    """
    s = series.dropna().values
    N = len(s)

    # Detrend by subtracting linear fit
    t      = np.arange(N)
    coeffs = np.polyfit(t, s, 1)
    detrended = s - np.polyval(coeffs, t)

    # FFT
    fft_vals = np.fft.rfft(detrended)
    freqs    = np.fft.rfftfreq(N)     # cycles per step
    power    = np.abs(fft_vals) ** 2

    # Skip frequency 0 (DC component)
    freqs = freqs[1:]
    power = power[1:]
    periods = 1.0 / freqs            # period in steps

    # Top-n dominant periods
    top_idx     = np.argsort(power)[::-1][:top_n]
    top_periods = periods[top_idx]
    top_power   = power[top_idx]

    # Filter the arrays for periods < N/2
    mask = periods < (N / 2)
    filtered_periods = periods[mask]
    filtered_Pxx = power[mask]

    # Plot
    # Power vs frequency plot

    # Power vs period (more interpretable)
    # Only show periods < N/2 (Nyquist)
    fig = make_subplots(rows=1, cols=2, subplot_titles=(f'Periodogram — {series_name}', f'Power vs Period — {series_name}'))
    fig.add_trace(go.Scatter(x=freqs, y=power, mode='lines', name='Power vs Frequency'), row=1, col=1)
    fig.add_trace(go.Scatter(x=filtered_periods, y=filtered_Pxx, mode='lines', name='Power vs Period'), row=1, col=2)
    for idx in top_idx:
        fig.add_vline(x=freqs[idx], line_dash='dash', line_color='black', row=1, col=1)
        fig.add_annotation(x=freqs[idx], y=power[idx], text=f' T≈{periods[idx]:.1f}', showarrow=True, arrowhead=1, ax=0, ay=-40, font_size=8, row=1, col=1)     
    fig.show()


    print(f"\nTop {top_n} dominant cycles in {series_name}:")
    print(f"  {'Rank':>4}  {'Period (steps)':>16}  {'Power':>12}  {'Freq':>10}")
    print("  " + "─"*48)
    for rank, (p, pw, f) in enumerate(zip(top_periods, top_power, freqs[top_idx]), 1):
        print(f"  {rank:>4}  {p:>16.2f}  {pw:>12.2f}  {f:>10.6f}")
    print()

day -1

In [19]:
plot_periodogram(mid_price_levels1,  series_name='Price Levels (detrended) day -1')
plot_periodogram(log_returns1, series_name='Log Returns day -1')


Top 5 dominant cycles in Price Levels (detrended) day -1:
  Rank    Period (steps)         Power        Freq
  ────────────────────────────────────────────────
     1           5000.00  861858797.51    0.000200
     2          10000.00  483057696.60    0.000100
     3           2500.00  318586222.38    0.000400
     4           1111.11  173208288.18    0.000900
     5            769.23  131864772.08    0.001300




Top 5 dominant cycles in Log Returns day -1:
  Rank    Period (steps)         Power        Freq
  ────────────────────────────────────────────────
     1              2.35          0.01    0.425943
     2              2.39          0.01    0.418842
     3              2.30          0.01    0.435644
     4              2.43          0.01    0.411241
     5              2.14          0.01    0.467547



day -2

In [20]:

plot_periodogram(mid_price_levels2,  series_name='Price Levels (detrended) day -2')
plot_periodogram(log_returns2, series_name='Log Returns day -2')


Top 5 dominant cycles in Price Levels (detrended) day -2:
  Rank    Period (steps)         Power        Freq
  ────────────────────────────────────────────────
     1           2500.00  1768749797.91    0.000400
     2          10000.00  724957713.16    0.000100
     3           5000.00  351596380.00    0.000200
     4           1428.57  186828162.63    0.000700
     5           1111.11  178663099.56    0.000900




Top 5 dominant cycles in Log Returns day -2:
  Rank    Period (steps)         Power        Freq
  ────────────────────────────────────────────────
     1              2.30          0.01    0.435244
     2              2.12          0.01    0.471147
     3              2.70          0.01    0.369737
     4              2.62          0.01    0.381138
     5              2.73          0.01    0.366737



### Interpretation guide

| Periodogram shape | Meaning |
|---|---|
| Flat, no dominant peak | No cyclical structure — consistent with white noise |
| One dominant sharp peak at period T | Clear cycle of length T steps |
| Multiple peaks at T, T/2, T/3 | Harmonic structure (strong fundamental cycle) |
| Power concentrated at low frequencies | Long, slow cycles / trend-like behaviour |
| Power decays as power law in frequency | Possible long-memory or fractal structure |

> **Competition implication:** A cycle of period T in returns means that on average, every T steps the price completes one mean-reversion cycle. This directly informs entry/exit timing — enter near cycle troughs, exit near peaks.
>
> **Caveat:** FFT peaks must be robust — always check if the same cycle appears in different sub-windows of the data before trading on it.

---
## Hurst Exponent: R/S Analysis

### Theory

The **Hurst Exponent H** characterises the *memory* and *self-similarity* of a time series.

| H | Process type | Behaviour |
|---|---|---|
| H = 0.5 | Random walk | No memory — efficient market |
| H > 0.5 | Persistent / trending | Trends persist — momentum strategies |
| H < 0.5 | Anti-persistent / mean-reverting | Reversals are more likely — mean-reversion strategies |

#### R/S (Rescaled Range) Method

For a window of size n:
1. Compute cumulative deviations from the mean: `Z_t = Σ(r_i - r̄)`
2. Range: `R = max(Z) - min(Z)`
3. Std dev: `S = std(r)`
4. Rescaled range: `R/S`

Repeat for many window sizes n. For a fractal process: `E[R/S] ~ c · n^H`

Taking logs: `log(R/S) = log(c) + H · log(n)`

So **H is the slope** of log(R/S) vs log(n) — we estimate it via linear regression.

In [21]:
def compute_hurst_rs(series, min_window=10, max_window=None, num_points=30):
    """
    Estimates the Hurst exponent using R/S analysis.
    Returns H, the log(n) values, log(R/S) values, and regression details.
    """
    s = np.array(series.dropna())
    N = len(s)
    if max_window is None:
        max_window = N // 2

    # Logarithmically spaced window sizes
    window_sizes = np.unique(
        np.logspace(np.log10(min_window), np.log10(max_window), num_points).astype(int)
    )

    rs_vals = []

    for n in window_sizes:
        # Split series into non-overlapping windows of size n
        n_chunks = N // n
        if n_chunks < 1:
            continue
        rs_chunk = []
        for i in range(n_chunks):
            chunk = s[i*n : (i+1)*n]
            mean  = chunk.mean()
            # Cumulative deviations from the mean
            cum_dev = np.cumsum(chunk - mean)
            R = cum_dev.max() - cum_dev.min()   # Range
            S = chunk.std(ddof=1)               # Std dev
            if S > 0:
                rs_chunk.append(R / S)
        if rs_chunk:
            rs_vals.append(np.mean(rs_chunk))
        else:
            window_sizes = window_sizes[window_sizes != n]

    log_n  = np.log(window_sizes[:len(rs_vals)])
    log_rs = np.log(rs_vals)

    # Linear regression: log(R/S) = a + H * log(n)
    slope, intercept, r_val, p_val, std_err = stats.linregress(log_n, log_rs)
    H = slope

    return H, log_n, log_rs, intercept, r_val**2, std_err

In [27]:

mid_price_levels1 = df1['mid_price']
log_returns1 = np.log(df1['mid_price']).diff().dropna()
H, log_n, log_rs, intercept, r2, std_err = compute_hurst_rs(log_returns1)

print("=" * 50)
print("  Hurst Exponent — R/S Analysis day -1")
print("=" * 50)
print(f"  H estimate  : {H:.4f}")
print(f"  Std error   : {std_err:.4f}")
print(f"  95% CI      : [{H - 1.96*std_err:.4f}, {H + 1.96*std_err:.4f}]")
print(f"  R² of fit   : {r2:.4f}")
print()
if H > 0.55:
    verdict = f'📈 TRENDING / PERSISTENT (H={H:.3f} > 0.5)'
elif H < 0.45:
    verdict = f'↩️  MEAN-REVERTING / ANTI-PERSISTENT (H={H:.3f} < 0.5)'
else:
    verdict = f'🎲 RANDOM WALK-LIKE (H={H:.3f} ≈ 0.5)'
print(f"  Verdict: {verdict}")
print("=" * 50)

  Hurst Exponent — R/S Analysis day -1
  H estimate  : 0.3876
  Std error   : 0.0040
  95% CI      : [0.3798, 0.3955]
  R² of fit   : 0.9970

  Verdict: ↩️  MEAN-REVERTING / ANTI-PERSISTENT (H=0.388 < 0.5)


In [28]:

mid_price_levels2 = df2['mid_price']
log_returns2 = np.log(df2['mid_price']).diff().dropna()
H, log_n, log_rs, intercept, r2, std_err = compute_hurst_rs(log_returns2)

print("=" * 50)
print("  Hurst Exponent — R/S Analysis day -2")
print("=" * 50)
print(f"  H estimate  : {H:.4f}")
print(f"  Std error   : {std_err:.4f}")
print(f"  95% CI      : [{H - 1.96*std_err:.4f}, {H + 1.96*std_err:.4f}]")
print(f"  R² of fit   : {r2:.4f}")
print()
if H > 0.55:
    verdict = f'📈 TRENDING / PERSISTENT (H={H:.3f} > 0.5)'
elif H < 0.45:
    verdict = f'↩️  MEAN-REVERTING / ANTI-PERSISTENT (H={H:.3f} < 0.5)'
else:
    verdict = f'🎲 RANDOM WALK-LIKE (H={H:.3f} ≈ 0.5)'
print(f"  Verdict: {verdict}")
print("=" * 50)

  Hurst Exponent — R/S Analysis day -2
  H estimate  : 0.3914
  Std error   : 0.0043
  95% CI      : [0.3830, 0.3999]
  R² of fit   : 0.9966

  Verdict: ↩️  MEAN-REVERTING / ANTI-PERSISTENT (H=0.391 < 0.5)


In [29]:
#Visualize R/S plot
def plot_rs_analysis(log_n, log_rs, intercept, H, series_name='Series'):
    # Fitted line
    fitted_log_rs = intercept + H * log_n

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=log_n, y=log_rs, mode='markers', name='Data (log(n), log(R/S))'))
    fig.add_trace(go.Scatter(x=log_n, y=fitted_log_rs, mode='lines', name=f'Fit: H={H:.3f}'))
    fig.update_layout(title=f'R/S Analysis for {series_name}', xaxis_title='log(Window Size)', yaxis_title='log(R/S)')
    fig.show()


plot_rs_analysis(log_n, log_rs, intercept, H, series_name='log_returns1')
plot_rs_analysis(log_n, log_rs, intercept, H, series_name='log_returns2')


### Interpretation guide

- **The log-log plot should be approximately linear.** If it curves, the Hurst exponent is not constant across scales — the process may have regime changes.
- **Compare the fitted slope to the H=0.5 dashed line.** Visually check whether your data clearly departs from the random-walk reference.
- **R² of the regression:** Should be high (> 0.95) for a reliable estimate. Low R² means the R/S scaling is not clean — be skeptical of H.
- **Confidence interval:** If the interval contains 0.5, you cannot statistically distinguish the series from a random walk.

> **Practical note:** R/S analysis is sensitive to short-range autocorrelation. For a more robust estimate, consider also using the `hurst` library which implements DFA (Detrended Fluctuation Analysis).

---
## Variance Ratio Test

### Theory

The **Variance Ratio (VR) test** (Lo & MacKinlay, 1988) tests whether a series is consistent with a random walk.

**Key insight:** For a random walk, variance grows *linearly* with time:
$$\text{Var}(r_t + r_{t+1} + ... + r_{t+k-1}) = k \cdot \text{Var}(r_t)$$

So the **variance ratio** is defined as:
$$\text{VR}(k) = \frac{\text{Var}(k\text{-period returns})}{k \cdot \text{Var}(1\text{-period returns})}$$

Under a random walk, VR(k) = 1 for all k.

| VR(k) > 1 | Positive autocorrelation at horizon k → **momentum** |
|---|---|
| VR(k) < 1 | Negative autocorrelation at horizon k → **mean-reversion** |
| VR(k) = 1 | Random walk — no exploitable structure |

Testing at *multiple horizons* k gives a richer picture: the series might be mean-reverting at short horizons but trending at long horizons.

In [ ]:
def variance_ratio(series, k):
    """
    Computes the variance ratio VR(k) and its z-statistic.
    Uses the heteroskedasticity-robust version (Lo-MacKinlay).

    Returns: VR estimate, z-statistic, p-value
    """
    r  = np.array(series.dropna())   # 1-period returns
    n  = len(r)
    mu = r.mean()

    # Variance of 1-period returns
    sigma2_1 = np.sum((r - mu)**2) / (n - 1)

    # k-period returns (overlapping)
    r_k = np.array([np.sum(r[t:t+k]) for t in range(n - k + 1)])

    # Variance of k-period returns (unbiased)
    m        = k * (n - k + 1) * (1 - k / n)
    sigma2_k = np.sum((r_k - k * mu)**2) / m

    VR = sigma2_k / (k * sigma2_1)

    # Heteroskedasticity-robust standard error (Lo-MacKinlay)
    delta = np.array([
        np.sum(
            (r[j:]  - mu)**2 * (r[j-lag:-lag] - mu)**2
            for j in range(lag, n)
        ).sum() / (np.sum((r - mu)**2))**2
        for lag in range(1, k)
    ])
    theta = np.sum([(1 - lag/k)**2 * delta[lag-1] for lag in range(1, k)])

    z_stat = (VR - 1) / np.sqrt(theta)
    p_val  = 2 * (1 - stats.norm.cdf(abs(z_stat)))

    return VR, z_stat, p_val


# Test at multiple horizons
horizons = [2, 4, 8, 16, 32]

print("=" * 65)
print("  Variance Ratio Test on Log Returns day -1")
print("  H₀: VR(k) = 1  (random walk at horizon k)")
print("=" * 65)
print(f"  {'k':>4}  {'VR(k)':>8}  {'z-stat':>10}  {'p-value':>10}  {'Interpretation':>20}")
print("  " + "─"*60)

vr_values = []
for k in horizons:
    try:
        vr, z, p = variance_ratio(log_returns1, k)
    except Exception:
        vr, z, p = np.nan, np.nan, np.nan
    vr_values.append(vr)
    if p < 0.05:
        interp = 'Momentum ↑' if vr > 1 else 'Mean-rev ↓'
    else:
        interp = 'Random walk'
    print(f"  {k:>4}  {vr:>8.4f}  {z:>10.4f}  {p:>10.4f}  {interp:>20}")

print("=" * 65)